In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

In [2]:
df = pd.read_csv('total_call_data.csv')

C:\Users\Owner\AppData\Local\Temp\ipykernel_17852\31442611.py:1: DtypeWarning: Columns (7) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv('total_call_data.csv')


In [4]:
df['Activity Name'].value_counts()[20:40]

Activity Name
PreTenantMenu                     15545
DivorceOrParentingMenu            14611
AppointmentMenu                   13576
OtherLegalOtherMenu               13067
HelpWithLegalorOtherReasonMenu    12098
PreQueueMessage2                  11488
LegalServerScreenPop              11369
FarmworkerMainMenu                 9832
ReadANI                            9213
DisconnectContact                  9121
CCB                                8754
PlayCCBConfirmation                8754
SuburbanSeniorsMenu                8642
BenefitsMenu                       7834
FrontDeskTransfer1                 7679
IntakePreQueueMessage1             7094
FrontDeskTransfer2                 6705
ImmigrationMenu                    5156
EmploymentMenu                     5068
SubSeniorPreQueueMessage1_1        4787
Name: count, dtype: int64

In [10]:
# Example target activity names
target_activities = ["PreQueueMessage1", "PreQueueMessage2", "QueueMenu1", "PlayMOH300s"]

# Assuming your DataFrame is named df and has at least these columns:
# "Call ID" (or whatever uniquely identifies a call) and "Activity Name"

# Step 1: Find all call IDs that hit one of the target activities
call_ids_to_keep = df.loc[df["Activity Name"].isin(target_activities), "Contact Session ID"].unique()

# Step 2: Filter to keep *all* rows of those calls
filtered_df = df[df["Contact Session ID"].isin(call_ids_to_keep)].copy().reset_index(drop = True)

In [73]:
menus_df

,Contact Session ID,Prev Activity
0,02f3875b-ceae-4836-9786-1daf78f2841b,LegalMenu2
1,04617841-2326-4535-bd4c-a9954f1cf4d7,SuburbanSeniorsMenu
2,06d5e9b7-9b3d-4a8a-9d50-7dfd63acaf45,SuburbanSeniorsMenu
3,07f88374-f76b-4b0a-819e-0de67976e69a,SuburbanSeniorsMenu
4,093223fc-33fe-48f0-aa19-f65535bfaec8,SuburbanSeniorsMenu
...,...,...
11483,c47cebb7-692a-495b-981c-a8ea7c2d2d71,ImmigrationMenu
11484,4b3b63de-f471-42be-a8a0-4ca70d57ea2b,TenantMenu
11485,289669ce-eab1-4a70-9aee-9eb7362642a2,BenefitsMenu
11486,da2bb927-df57-46c7-9a34-0e973c62fd3f,BenefitsMenu


In [74]:
filtered_df.loc[filtered_df['Contact Session ID'] == '02f3875b-ceae-4836-9786-1daf78f2841b']   

,Contact Session ID,EP Name,Flow Name,Activity Name,Activity Start Timestamp,Queue Name,Agent Name,Termination Reason,hour
0,02f3875b-ceae-4836-9786-1daf78f2841b,Main Number Telephony EP,NaN,NaN,2025-01-17 08:01:09,NaN,NaN,NaN,8
1,02f3875b-ceae-4836-9786-1daf78f2841b,NaN,LACMain,NaN,2025-01-17 08:01:09,NaN,NaN,NaN,8
2,02f3875b-ceae-4836-9786-1daf78f2841b,Main Number Telephony EP,NaN,LanguageSelectionMenu,2025-01-17 08:01:09,NaN,NaN,NaN,8
3,02f3875b-ceae-4836-9786-1daf78f2841b,Main Number Telephony EP,LACMain,NaN,2025-01-17 08:01:09,NaN,NaN,NaN,8
4,02f3875b-ceae-4836-9786-1daf78f2841b,Main Number Telephony EP,NaN,MainMenu,2025-01-17 08:01:21,NaN,NaN,NaN,8
5,02f3875b-ceae-4836-9786-1daf78f2841b,NaN,PreLegalMenuSeniorsMenu,NaN,2025-01-17 08:01:45,NaN,NaN,NaN,8
6,02f3875b-ceae-4836-9786-1daf78f2841b,Pre-Legal Menu Seniors Menu Telephony EP,NaN,SeniorsMenu,2025-01-17 08:01:45,NaN,NaN,NaN,8
7,02f3875b-ceae-4836-9786-1daf78f2841b,NaN,LegalMenu,NaN,2025-01-17 08:01:53,NaN,NaN,NaN,8
8,02f3875b-ceae-4836-9786-1daf78f2841b,Legal Menu Telephony EP,NaN,LegalMenu1,2025-01-17 08:01:53,NaN,NaN,NaN,8
9,02f3875b-ceae-4836-9786-1daf78f2841b,Legal Menu Telephony EP,NaN,LegalMenu2,2025-01-17 08:02:29,NaN,NaN,NaN,8


In [28]:
# This code tells use which menus lead to the most queues
results = []

for contact_id in filtered_df['Contact Session ID'].unique():
    temp = filtered_df.loc[filtered_df['Contact Session ID'] == contact_id].copy()
    temp = temp.sort_values('Activity Start Timestamp').reset_index(drop=True)
    
    mask = temp['Activity Name'].str.contains('queue', case=False, na=False) | \
           temp['Flow Name'].str.contains('queue', case=False, na=False)
    
    queue_rows = temp[mask]
    if not queue_rows.empty:
        first_queue_index = queue_rows.index[0]
        if first_queue_index > 0:
            prev_activity = temp.loc[first_queue_index - 1, 'Activity Name']
            results.append({'Contact Session ID': contact_id, 'Prev Activity': prev_activity})

menus_df = pd.DataFrame(results)

In [79]:
# Uses Queue Name
# create queue_name_df similar to menus_df but using 'Queue Name' instead of 'Activity Name'
results = []


for contact_id in filtered_df['Contact Session ID'].unique():
    temp = filtered_df.loc[filtered_df['Contact Session ID'] == contact_id].copy()
    temp = temp.sort_values('Activity Start Timestamp').reset_index(drop=True)
    
    for i in range(1, len(temp)):
        current_row = temp.iloc[i]
        prev_row = temp.iloc[i - 1]
        queue_name = current_row['Queue Name']
        if queue_name not in [np.nan, '']:
            
            results.append({'Contact Session ID': contact_id, 'Queue Name': queue_name})
            break
queue_name_df = pd.DataFrame(results)   

In [80]:
queue_name_df['Queue Name'].value_counts()

Queue Name
Family                      1874
Consumer                    1399
SubSenior Other             1158
Housing                     1007
Benefits                     773
SubSenior Benefits           662
SubSenior Tenant             656
SubSenior Consumer           497
Employment                   492
SubSenior Family             462
SubSenior Homeowner          440
ADAPT                        368
SubSenior ADAPT              255
Family SP                    254
Immigration SP               164
SubSenior Employment         143
Consumer SP                  113
Benefits SP                   95
Education                     82
Employment SP                 81
SubSenior Other SP            62
Immigration                   55
SubSenior Consumer SP         44
SubSenior Benefits SP         44
Other SubSeniors              33
ADAPT SP                      28
SubSenior Family SP           27
SubSenior Homeowner SP        22
Housing SP                    21
Benefits SubSeniors           19

In [ ]:
# Uses Activity Name

results = []

for contact_id in filtered_df['Contact Session ID'].unique():
    temp = filtered_df.loc[filtered_df['Contact Session ID'] == contact_id].copy()
    temp = temp.sort_values('Activity Start Timestamp').reset_index(drop=True)
    
    mask = temp['Activity Name'].str.contains('queue', case=False, na=False) | \
           temp['Flow Name'].str.contains('queue', case=False, na=False)
    
    queue_rows = temp[mask]
    if not queue_rows.empty:
        first_queue_index = queue_rows.index[0]
        
        # Walk backward to find the most recent non-Na activity not starting with GetLoggedIn
        prev_activity = None
        for i in range(first_queue_index - 1, -1, -1):  # iterate backward
            activity = temp.loc[i, 'Activity Name']
            if pd.notna(activity) and not str(activity).startswith('GetLoggedIn'):
                prev_activity = activity
                break
        
        # Only record if we actually found a valid previous activity
        if prev_activity:
            results.append({
                'Contact Session ID': contact_id,
                'Prev Activity': prev_activity
            })

menus_df = pd.DataFrame(results)

In [68]:
# Here we see that SuburbanSeniorsMenu has almost 3 times as many queue leads as the next highest
# However, this could be skewed if some menus are named differently or are not correclty grouped together
menus_df['Prev Activity'].value_counts()

Prev Activity
SuburbanSeniorsMenu       3973
FamilyMenu                1362
LegalMenu2                1089
TenantMenu                1028
BenefitsMenu               868
DivorceOrParentingMenu     776
PreTenantMenu              687
HousingMenu                594
EmploymentMenu             573
SeniorsADAPTMenu           221
ImmigrationMenu            219
SimpleDivorceMenu           86
SuburbsOrCityMenu            8
OtherLegalMenu               2
LegalMenu1                   1
TenantDeterrenceMenu         1
Name: count, dtype: int64

In [ ]:
queue_times = {}

results = []

for contact_id in filtered_df['Contact Session ID'].unique():
    temp = filtered_df.loc[filtered_df['Contact Session ID'] == contact_id].copy()
    temp = temp.sort_values('Activity Start Timestamp').reset_index(drop=True)
    
    mask = temp['Activity Name'].str.contains('queue', case=False, na=False) | \
           temp['Flow Name'].str.contains('queue', case=False, na=False)
    
    queue_rows = temp[mask]
    if not queue_rows.empty:
        first_queue_index = queue_rows.index[0]
        
        # Walk backward to find the most recent non-Na activity not starting with GetLoggedIn
        prev_activity = None
        for i in range(first_queue_index - 1, -1, -1):  # iterate backward
            activity = temp.loc[i, 'Activity Name']
            if pd.notna(activity) and not str(activity).startswith('GetLoggedIn'):
                prev_activity = activity
                break
        
        # start recording time in queue when we hit PlayMOH300s
        # Record every instance of PlayMOH300s and QueueMenu1 until we exit the queue
        occurences = 0
        for i in range(first_queue_index, len(temp)):
            activity = temp.loc[i, 'Activity Name']
            if activity == 'PlayMOH300s' or activity == 'QueueMenu1':
                occurences += 1
                
                
                # Accumulate time for this previous activity
                if prev_activity not in queue_times:
                    queue_times[prev_activity] = 0
                queue_times[prev_activity] += occurences
            else:
                # Exit the queue when we hit an activity that is not part of the queue flow
                break
for key, value in queue_times.items():
       total = queue_name_df['Queue Name'].value_counts().get(key, 1)



['LegalMenu2',
 'SuburbanSeniorsMenu',
 'DivorceOrParentingMenu',
 'FamilyMenu',
 'HousingMenu',
 'BenefitsMenu',
 'SeniorsADAPTMenu',
 'PreTenantMenu',
 'EmploymentMenu',
 'TenantMenu',
 'ImmigrationMenu',
 'SimpleDivorceMenu',
 'SuburbsOrCityMenu',
 'LegalMenu1',
 'OtherLegalMenu',
 'TenantDeterrenceMenu']

In [71]:
menus_df['Prev Activity'].value_counts().get('LegalMenu2', 1)

np.int64(1089)

In [62]:
filtered_df.loc[filtered_df['Contact Session ID'] == 'c47cebb7-692a-495b-981c-a8ea7c2d2d71']

,Contact Session ID,EP Name,Flow Name,Activity Name,Activity Start Timestamp,Queue Name,Agent Name,Termination Reason,hour
408105,c47cebb7-692a-495b-981c-a8ea7c2d2d71,Main Number Telephony EP,NaN,NaN,2025-03-28 08:08:15,NaN,NaN,NaN,8
408106,c47cebb7-692a-495b-981c-a8ea7c2d2d71,NaN,LACMain,NaN,2025-03-28 08:08:15,NaN,NaN,NaN,8
408107,c47cebb7-692a-495b-981c-a8ea7c2d2d71,Main Number Telephony EP,NaN,LanguageSelectionMenu,2025-03-28 08:08:15,NaN,NaN,NaN,8
408108,c47cebb7-692a-495b-981c-a8ea7c2d2d71,Main Number Telephony EP,LACMain,NaN,2025-03-28 08:08:15,NaN,NaN,NaN,8
408111,c47cebb7-692a-495b-981c-a8ea7c2d2d71,Main Number Telephony EP,NaN,MainMenu,2025-03-28 08:08:23,NaN,NaN,NaN,8
408124,c47cebb7-692a-495b-981c-a8ea7c2d2d71,NaN,PreLegalMenuSeniorsMenu,NaN,2025-03-28 08:08:53,NaN,NaN,NaN,8
408125,c47cebb7-692a-495b-981c-a8ea7c2d2d71,Pre-Legal Menu Seniors Menu Telephony EP,NaN,SeniorsMenu,2025-03-28 08:08:53,NaN,NaN,NaN,8
408138,c47cebb7-692a-495b-981c-a8ea7c2d2d71,Pre-Legal Menu Seniors Menu Telephony EP,NaN,SeniorsMenu,2025-03-28 08:09:06,NaN,NaN,NaN,8
408141,c47cebb7-692a-495b-981c-a8ea7c2d2d71,NaN,LegalMenu,NaN,2025-03-28 08:09:09,NaN,NaN,NaN,8
408142,c47cebb7-692a-495b-981c-a8ea7c2d2d71,Legal Menu Telephony EP,NaN,LegalMenu1,2025-03-28 08:09:09,NaN,NaN,NaN,8


In [13]:
temp = df.loc[(df['Activity Name'] == 'QueueMenu1') | (df['Activity Name'] == 'PlayMOH300s')].reset_index()

In [18]:
record = 0
for i, row in temp.iterrows():
    if i == 0:
        continue
    if row['Activity Name'] == 'PlayMOH300s':
        if temp.loc[i-1, 'Activity Name'] != 'QueueMenu1':
            print(temp.loc[i-1, 'Activity Name'])
            record += 1
            print(i)

PlayMOH300s
7
PlayMOH300s
8
PlayMOH300s
53
PlayMOH300s
54
PlayMOH300s
111
PlayMOH300s
114
PlayMOH300s
131
PlayMOH300s
282
PlayMOH300s
333
PlayMOH300s
340
PlayMOH300s
359
PlayMOH300s
374
PlayMOH300s
427
PlayMOH300s
464
PlayMOH300s
635
PlayMOH300s
636
PlayMOH300s
637
PlayMOH300s
638
PlayMOH300s
661
PlayMOH300s
687
PlayMOH300s
692
PlayMOH300s
693
PlayMOH300s
700
PlayMOH300s
739
PlayMOH300s
794
PlayMOH300s
820
PlayMOH300s
824
PlayMOH300s
900
PlayMOH300s
901
PlayMOH300s
902
PlayMOH300s
903
PlayMOH300s
904
PlayMOH300s
905
PlayMOH300s
906
PlayMOH300s
907
PlayMOH300s
910
PlayMOH300s
911
PlayMOH300s
912
PlayMOH300s
923
PlayMOH300s
930
PlayMOH300s
941
PlayMOH300s
949
PlayMOH300s
957
PlayMOH300s
965
PlayMOH300s
972
PlayMOH300s
978
PlayMOH300s
984
PlayMOH300s
990
PlayMOH300s
1012
PlayMOH300s
1018
PlayMOH300s
1025
PlayMOH300s
1031
PlayMOH300s
1045
PlayMOH300s
1057
PlayMOH300s
1068
PlayMOH300s
1086
PlayMOH300s
1091
PlayMOH300s
1122
PlayMOH300s
1128
PlayMOH300s
1129
PlayMOH300s
1135
PlayMOH300s
1138


In [31]:
df.loc[1228:1258]

,Contact Session ID,EP Name,Flow Name,Activity Name,Activity Start Timestamp,Queue Name,Agent Name,Termination Reason,hour
1228,06d5e9b7-9b3d-4a8a-9d50-7dfd63acaf45,Pre-Legal Menu Seniors Menu Telephony EP,NaN,SuburbsOrCityMenu,2025-01-14 13:12:51,NaN,NaN,NaN,13
1229,06d5e9b7-9b3d-4a8a-9d50-7dfd63acaf45,Pre-Legal Menu Seniors Menu Telephony EP,NaN,SuburbanSeniorsMenu,2025-01-14 13:13:09,NaN,NaN,NaN,13
1230,06d5e9b7-9b3d-4a8a-9d50-7dfd63acaf45,NaN,Queues,NaN,2025-01-14 13:14:48,NaN,NaN,NaN,13
1231,06d5e9b7-9b3d-4a8a-9d50-7dfd63acaf45,All LAC Queues Telephony EP,NaN,GetLoggedInSubSeniorOtherAgents,2025-01-14 13:14:48,NaN,NaN,NaN,13
1232,06d5e9b7-9b3d-4a8a-9d50-7dfd63acaf45,All LAC Queues Telephony EP,NaN,SubSeniorPreQueueMessage1_1,2025-01-14 13:14:48,NaN,NaN,NaN,13
1233,06d5e9b7-9b3d-4a8a-9d50-7dfd63acaf45,All LAC Queues Telephony EP,NaN,SubSeniorOtherQueue,2025-01-14 13:15:54,NaN,NaN,NaN,13
1234,06d5e9b7-9b3d-4a8a-9d50-7dfd63acaf45,NaN,NaN,NaN,2025-01-14 13:15:54,SubSenior Other,NaN,NaN,13
1235,06d5e9b7-9b3d-4a8a-9d50-7dfd63acaf45,All LAC Queues Telephony EP,NaN,NaN,2025-01-14 13:15:54,NaN,NaN,NaN,13
1236,06d5e9b7-9b3d-4a8a-9d50-7dfd63acaf45,All LAC Queues Telephony EP,NaN,NaN,2025-01-14 13:15:54,SubSenior Other,NaN,NaN,13
1237,06d5e9b7-9b3d-4a8a-9d50-7dfd63acaf45,All LAC Queues Telephony EP,NaN,PreQueueMessage2,2025-01-14 13:15:54,NaN,NaN,NaN,13


In [20]:
temp.loc[4:10]

,index,Contact Session ID,EP Name,Flow Name,Activity Name,Activity Start Timestamp,Queue Name,Agent Name,Termination Reason,hour
4,1240,06d5e9b7-9b3d-4a8a-9d50-7dfd63acaf45,All LAC Queues Telephony EP,NaN,PlayMOH300s,2025-01-14 13:21:28,NaN,NaN,NaN,13
5,1241,06d5e9b7-9b3d-4a8a-9d50-7dfd63acaf45,All LAC Queues Telephony EP,NaN,QueueMenu1,2025-01-14 13:26:29,NaN,NaN,NaN,13
6,1242,06d5e9b7-9b3d-4a8a-9d50-7dfd63acaf45,All LAC Queues Telephony EP,NaN,PlayMOH300s,2025-01-14 13:26:50,NaN,NaN,NaN,13
7,1248,06d5e9b7-9b3d-4a8a-9d50-7dfd63acaf45,All LAC Queues Telephony EP,NaN,PlayMOH300s,2025-01-14 13:29:39,NaN,NaN,NaN,13
8,1743,093223fc-33fe-48f0-aa19-f65535bfaec8,All LAC Queues Telephony EP,NaN,PlayMOH300s,2025-01-14 15:20:03,NaN,NaN,NaN,15
9,1744,093223fc-33fe-48f0-aa19-f65535bfaec8,All LAC Queues Telephony EP,NaN,QueueMenu1,2025-01-14 15:25:04,NaN,NaN,NaN,15
10,2331,0bfab117-b496-45c8-9ead-31a176aa6a95,All LAC Queues Telephony EP,NaN,PlayMOH300s,2025-01-13 08:24:19,NaN,NaN,NaN,8


In [6]:
df['Flow Name'].value_counts()

Flow Name
LACMain                    438461
PreLegalMenuSeniorsMenu    114943
LegalMenu                  103845
Queues                      60361
ClosedQueueMenu             50752
LegalFamilyMenu             25085
FarmworkerMain              21376
ClosedHoursHolidaysMenu     21290
LegalHousingMenu            19875
OtherLegalMenu              14880
Intake_Outdial               9126
CourtesyCallback             8930
LegalBenefitsMenu            7259
LegalEmploymentMenu          4559
LegalImmigrationMenu         3616
LegalHIVMenu                 2174
Name: count, dtype: int64